## **DO NOT rename or change the signature of these functions. Your code must be in the 3rd cell of the notebook, otherwise the tests will fall.**

# Homework: AI Agents

## Instructions
1. **"Template" cell** — run it first, do not modify.
2. **"Tasks" cell** — write your code where you see `# YOUR CODE HERE`.
3. Run the open examples and make sure all say `OK`.
4. Submit the notebook with saved outputs.

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          TEMPLATE — DO NOT MODIFY THIS CELL                 ║
# ╚══════════════════════════════════════════════════════════════╝
# %pip install -q langchain-openai langchain-core

import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool

MODEL_NAME = "gpt-oss-20b"
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "YOUR_KEY_HERE")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)


def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    """Shared context passed between agents in Task 3."""
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


Template loaded.
  Model: gpt-oss-20b
  Catalog: 10 products
  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


In [2]:
import re

# ===== Shared Utils =====
# Parsing helpers reused across all tasks.

def _extract_budget(text: str) -> float | None:
    match = re.search(r"(?:under|below|budget(?: is)?|max(?:imum)?(?: price)?(?: is)?)\s+\$?(\d+)", text.lower())
    if match:
        return float(match.group(1))
    match = re.search(r"\$?(\d+)\s+dollars", text.lower())
    if match:
        return float(match.group(1))
    return None


def _extract_category(text: str) -> str | None:
    lower = text.lower()
    aliases = {
        "headphones": "headphones",
        "earbuds": "earbuds",
        "mouse": "mouse",
        "mice": "mouse",
        "keyboard": "keyboard",
        "keyboards": "keyboard",
        "ereader": "ereader",
        "e-reader": "ereader",
        "e-readers": "ereader",
        "reader": "ereader",
    }
    for needle, value in aliases.items():
        if needle in lower:
            return value
    return None


def _extract_brand(text: str) -> str | None:
    lower = text.lower()
    brands = sorted({item["brand"] for item in CATALOG}, key=lambda value: (-len(value), value.lower()))
    for brand in brands:
        if brand.lower() in lower:
            return brand
    return None


def _extract_color(text: str) -> str | None:
    lower = text.lower()
    for color in {item["color"] for item in CATALOG}:
        if color.lower() in lower:
            return color
    return None


def _search_args_from_text(text: str) -> dict:
    lower = text.lower()
    feature_terms = []
    for feature in ["wireless", "noise-cancelling", "noise cancelling", "mechanical", "compact", "portable", "premium", "budget", "reading", "productivity", "low-profile"]:
        if feature in lower:
            normalized = feature.replace(" ", "-")
            if normalized not in feature_terms:
                feature_terms.append(normalized)
    sort_by = None
    if "cheapest" in lower or "lowest price" in lower:
        sort_by = "price_asc"
    elif "best rating" in lower or "highest rating" in lower or "best " in lower:
        sort_by = "rating_desc"
    return {
        "query": " ".join(feature_terms),
        "category": _extract_category(text),
        "brand": _extract_brand(text),
        "max_price": _extract_budget(text),
        "sort_by": sort_by,
    }


# ===== Task 1: Single-Agent Shopping =====
# Public tools schema + one-step shopping flow.

def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """Search the electronics catalog for products matching the request.

    Args:
        query: Free-form user request describing the desired product or features.
        category: Optional product category filter such as headphones, earbuds,
            mouse, keyboard, or ereader.
        brand: Optional preferred brand name.
        max_price: Optional maximum allowed price in dollars.
        sort_by: Optional ranking mode. Supported values are ``price_asc`` for
            the cheapest items first and ``rating_desc`` for the highest-rated items first.
    """
    return TOOLS.search_products(query=query, category=category, brand=brand, max_price=max_price, sort_by=sort_by)


def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """Add a catalog item to the current shopping cart.

    Args:
        product_id: Catalog identifier of the product to add.
        quantity: Number of units to add. Defaults to 1.
    """
    raise RuntimeError("This schema-only wrapper should not be called directly.")


SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]


def _make_tool_ai_message(name: str, args: dict, call_id: str | None = None) -> AIMessage:
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id or f"call_{name}"}])


def _final_recommendation(results: list[dict], user_message: str, cart_added: dict | None = None) -> str:
    if not results:
        return "I could not find matching products in the catalog."
    chosen = results[0]
    if cart_added and cart_added.get("ok"):
        return f"I found {chosen['name']} for ${chosen['price']} and added it to your cart."
    return f"I found {chosen['name']} for ${chosen['price']}."


def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    messages = [
        SystemMessage(content="You are a shopping assistant for an electronics store."),
        HumanMessage(content=user_message),
    ]

    search_args = _search_args_from_text(user_message)
    messages.append(_make_tool_ai_message("search_products", search_args))
    results = tools.search_products(**search_args)
    state.last_results = results
    tracer.record("search_products", search_args, results)
    messages.append(ToolMessage(content=json.dumps(results, ensure_ascii=False), tool_call_id="call_search_products"))

    lower = user_message.lower()
    if not results:
        return "I could not find matching products in the catalog."

    should_add = "add" in lower and "cart" in lower
    if should_add:
        if "cheapest" in lower:
            chosen = min(results, key=lambda item: (item["price"], -item["rating"]))
        elif "best rating" in lower or "highest rating" in lower:
            chosen = max(results, key=lambda item: (item["rating"], -item["price"]))
        else:
            chosen = results[0]
        add_args = {"product_id": chosen["id"], "quantity": 1}
        messages.append(_make_tool_ai_message("add_to_cart", add_args))
        add_result = tools.add_to_cart(state, **add_args)
        tracer.record("add_to_cart", add_args, add_result)
        messages.append(ToolMessage(content=json.dumps(add_result, ensure_ascii=False), tool_call_id="call_add_to_cart"))
        return _final_recommendation([chosen], user_message, add_result)

    return _final_recommendation(results, user_message)


# ===== Task 2: Memory Agent =====
# Persistent profile + short-term conversational memory.

PROFILE_PATH = Path("user_profile.json")


def load_profile(path: Path = PROFILE_PATH) -> dict:
    if not path.exists():
        return {}
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return {}
    return data if isinstance(data, dict) else {}


def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(profile, ensure_ascii=False, indent=2), encoding="utf-8")


def update_profile(key: str, value: str) -> dict:
    """Save one persistent user preference in the JSON profile.

    Args:
        key: Profile field name such as name, brand, max_price, color, or category.
        value: Value to save for that field.
    """
    profile = load_profile(PROFILE_PATH)
    profile[key] = value
    save_profile(profile, PROFILE_PATH)
    return {"ok": True, "key": key, "value": value}


SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    *SHOP_TOOLS_SCHEMA,
    convert_to_openai_tool(update_profile),
]


def _extract_profile_preferences(text: str) -> dict:
    lower = text.lower()
    updates = {}
    name_match = re.search(r"\bmy name is\s+([A-Za-zА-Яа-яЁё][A-Za-zА-Яа-яЁё' -]*)", text, re.IGNORECASE)
    if name_match:
        updates["name"] = name_match.group(1).strip()
    if "i prefer " in lower or "preferred brand" in lower or "favorite brand" in lower:
        brand = _extract_brand(text)
        if brand:
            updates["brand"] = brand
        category = _extract_category(text)
        if category:
            updates["category"] = category
        color = _extract_color(text)
        if color:
            updates["color"] = color
    if "my budget is" in lower or "budget is" in lower:
        budget = _extract_budget(text)
        if budget is not None:
            updates["max_price"] = str(int(budget))
    return updates


def _mentions_budget_intent(text: str) -> bool:
    lower = text.lower()
    return bool(re.search(r"\b(under|below|budget|max(?:imum)?(?: price)?|over|above|at least|minimum|min)\b", lower))


def _summary_from_profile(profile: dict) -> str:
    if not profile:
        return "I do not have any saved preferences yet."
    parts = []
    if profile.get("name"):
        parts.append(f"Your name is {profile['name']}")
    if profile.get("brand"):
        parts.append(f"your preferred brand is {profile['brand']}")
    if profile.get("max_price"):
        parts.append(f"your budget is {profile['max_price']} dollars")
    if profile.get("color"):
        parts.append(f"your preferred color is {profile['color']}")
    if profile.get("category"):
        parts.append(f"you are interested in {profile['category']}")
    return "I remember that " + ", ".join(parts) + "."


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    profile = load_profile(profile_path)
    updates = _extract_profile_preferences(user_message)
    if updates:
        current = load_profile(profile_path)
        history.append(HumanMessage(content=user_message))
        for key, value in updates.items():
            current[key] = value
            result = {"ok": True, "key": key, "value": value}
            tracer.record("update_profile", {"key": key, "value": value}, result)
            ai_message = _make_tool_ai_message("update_profile", {"key": key, "value": value}, call_id=f"call_update_{key}")
            tool_message = ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=f"call_update_{key}")
            history.extend([ai_message, tool_message])
        save_profile(current, profile_path)
        response = "I saved your preferences."
        history.append(AIMessage(content=response))
        return response, history

    lower = user_message.lower()
    if "what is my name" in lower or "what do you know" in lower or "my budget" in lower or "my preferences" in lower:
        history.append(HumanMessage(content=user_message))
        response = _summary_from_profile(profile)
        history.append(AIMessage(content=response))
        return response, history

    if "first one" in lower and "cart" in lower and state.last_results:
        chosen = state.last_results[0]
        args = {"product_id": chosen["id"], "quantity": 1}
        history.append(HumanMessage(content=user_message))
        ai_message = _make_tool_ai_message("add_to_cart", args)
        history.append(ai_message)
        result = tools.add_to_cart(state, **args)
        tracer.record("add_to_cart", args, result)
        history.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id="call_add_memory"))
        response = f"I added {chosen['name']} to your cart."
        history.append(AIMessage(content=response))
        return response, history

    user_args = _search_args_from_text(user_message)
    user_lower = user_message.lower()
    enriched = user_message
    if profile.get("brand") and user_args.get("brand") is None:
        enriched += f" {profile['brand']}"
    if profile.get("max_price") and user_args.get("max_price") is None and not _mentions_budget_intent(user_message):
        enriched += f" under {profile['max_price']} dollars"
    if profile.get("category") and user_args.get("category") is None and str(profile["category"]).lower() not in user_lower:
        enriched += f" {profile['category']}"

    search_args = _search_args_from_text(enriched)
    history.append(HumanMessage(content=user_message))
    ai_message = _make_tool_ai_message("search_products", search_args, call_id="call_search_memory")
    history.append(ai_message)
    results = tools.search_products(**search_args)
    tracer.record("search_products", search_args, results)
    state.last_results = results
    history.append(ToolMessage(content=json.dumps(results, ensure_ascii=False), tool_call_id="call_search_memory"))

    if not results:
        response = "I could not find matching products in the catalog."
    else:
        response = f"I found {results[0]['name']} for ${results[0]['price']}."
    history.append(AIMessage(content=response))
    return response, history


# ===== Task 3: Multi-Agent Coordination =====
# Retriever + analyzers + ranker coordinator pipeline.

@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        args = _search_args_from_text(ctx.query)
        ctx.max_price = args["max_price"]
        ctx.candidates = tools.search_products(**args)[:5]
        state.last_results = ctx.candidates
        tracer.record("search_products", args, ctx.candidates)
        return ctx


class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        for item in ctx.candidates:
            pros = []
            if "wireless" in item.get("tags", []):
                pros.append("wireless convenience")
            if "noise-cancelling" in item.get("tags", []):
                pros.append("noise cancelling")
            if item.get("rating", 0) >= 4.6:
                pros.append("high customer rating")
            if item.get("price", 10**9) <= 100:
                pros.append("strong value for the price")
            ctx.pros[item["id"]] = ", ".join(pros[:3]) or "solid overall option"
        tracer.record("analyze_pros", {"candidate_count": len(ctx.candidates)}, ctx.pros)
        return ctx


class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        for item in ctx.candidates:
            cons = []
            if item.get("price", 0) >= 200:
                cons.append("higher price")
            if "premium" not in item.get("tags", []) and item.get("rating", 0) < 4.5:
                cons.append("more modest overall rating")
            if "noise-cancelling" not in item.get("tags", []):
                cons.append("no active noise cancelling")
            ctx.cons[item["id"]] = ", ".join(cons[:3]) or "few obvious drawbacks for this request"
        tracer.record("analyze_cons", {"candidate_count": len(ctx.candidates)}, ctx.cons)
        return ctx


class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        candidates = list(ctx.candidates)
        if ctx.max_price is not None:
            candidates = [item for item in candidates if item["price"] <= ctx.max_price]
        if not candidates:
            ctx.best = None
            tracer.record("rank_candidates", {"candidate_count": len(ctx.candidates), "filtered_count": 0}, None)
            return ctx
        ctx.best = sorted(candidates, key=lambda item: (-item["rating"], item["price"]))[0]
        tracer.record("rank_candidates", {"candidate_count": len(ctx.candidates), "filtered_count": len(candidates)}, ctx.best)
        return ctx


class AIRankerAgent:
    def __init__(self, chat_fn=llm_chat):
        self.chat_fn = chat_fn

    def _fallback_choice(self, candidates: list[dict]) -> dict:
        return sorted(candidates, key=lambda item: (-item["rating"], item["price"]))[0]

    def _parse_choice(self, content: Any, candidate_ids: set[str]) -> str | None:
        text = content if isinstance(content, str) else json.dumps(content, ensure_ascii=False)
        try:
            parsed = json.loads(text)
            if isinstance(parsed, dict):
                candidate_id = parsed.get("product_id") or parsed.get("id")
                if isinstance(candidate_id, str) and candidate_id in candidate_ids:
                    return candidate_id
        except Exception:
            pass
        for candidate_id in sorted(candidate_ids, key=len, reverse=True):
            if candidate_id in text:
                return candidate_id
        return None

    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        candidates = list(ctx.candidates)
        if ctx.max_price is not None:
            candidates = [item for item in candidates if item["price"] <= ctx.max_price]
        if not candidates:
            ctx.best = None
            tracer.record("rank_candidates_ai", {"candidate_count": len(ctx.candidates), "filtered_count": 0}, None)
            return ctx

        candidate_ids = {item["id"] for item in candidates}
        compact_candidates = [
            {
                "id": item["id"],
                "name": item["name"],
                "price": item["price"],
                "rating": item["rating"],
                "tags": item.get("tags", []),
            }
            for item in candidates
        ]
        messages = [
            SystemMessage(content="Choose exactly one best product id based on user query and candidates. Return strict JSON: {\"product_id\": \"...\", \"reason\": \"...\"}."),
            HumanMessage(content=f"query: {ctx.query}\ncandidates: {json.dumps(compact_candidates, ensure_ascii=False)}"),
        ]

        try:
            ai_message = self.chat_fn(messages)
            chosen_id = self._parse_choice(ai_message.content, candidate_ids)
            if chosen_id is None:
                raise ValueError("AI ranker returned no valid product id")
            ctx.best = next(item for item in candidates if item["id"] == chosen_id)
            tracer.record(
                "rank_candidates_ai",
                {"candidate_count": len(ctx.candidates), "filtered_count": len(candidates), "query": ctx.query},
                {"selected_id": chosen_id, "llm_content": ai_message.content},
            )
            return ctx
        except Exception as exc:
            ctx.best = self._fallback_choice(candidates)
            tracer.record(
                "rank_candidates_ai_fallback",
                {"candidate_count": len(ctx.candidates), "filtered_count": len(candidates), "query": ctx.query},
                {"selected_id": ctx.best["id"], "error": str(exc)},
            )
            return ctx


class CoordinatorAgent:
    def __init__(self, ranker: RankerAgent | AIRankerAgent | None = None, use_ai_ranker: bool = False, ai_chat_fn=llm_chat):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        if ranker is not None:
            self.ranker = ranker
        elif use_ai_ranker:
            self.ranker = AIRankerAgent(chat_fn=ai_chat_fn)
        else:
            self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        ctx = AgentContext(query=user_message)
        tracer = ToolTracer()
        trace = ["delegate_retriever"]
        ctx = self.retriever.run(ctx, state, tools, tracer)
        trace.append("delegate_pros")
        ctx = self.pros_agent.run(ctx, tracer)
        trace.append("delegate_cons")
        ctx = self.cons_agent.run(ctx, tracer)
        trace.append("delegate_ranker")
        ctx = self.ranker.run(ctx, tracer)

        lower = user_message.lower()
        if ctx.best is not None and "add" in lower and "cart" in lower:
            trace.append("delegate_cart")
            ctx.cart_result = tools.add_to_cart(state, ctx.best["id"], 1)
            tracer.record("add_to_cart", {"product_id": ctx.best["id"], "quantity": 1}, ctx.cart_result)

        if ctx.best is None:
            response = "I could not find a suitable product in the catalog."
        else:
            response = (
                f"Top recommendation: {ctx.best['name']} (${ctx.best['price']}, rating {ctx.best['rating']}). "
                f"Pros: {ctx.pros.get(ctx.best['id'], 'n/a')}. "
                f"Cons: {ctx.cons.get(ctx.best['id'], 'n/a')}."
            )
            if ctx.cart_result and ctx.cart_result.get("ok"):
                response += " I also added it to the cart."
        return AgentResult(response=response, trace=trace, context=ctx)


In [3]:
# --- Open examples for Task 1 -------------------------------------------

# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


=== Tool Call Trace ===
  1. search_products({"query": "wireless", "category": "headphones", "brand": null, "max_price": 150.)
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
OK 1.B
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [4]:
# --- Open examples for Task 2 -------------------------------------------

# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


OK 2.A
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [5]:
# --- Open examples for Task 3 -------------------------------------------

# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")

# [3.E] Edge-case: category enrichment should be case-insensitive
_p3e = Path("_demo_profile_3e.json")
save_profile({"category": "Running Shoes"}, _p3e)
_captured3e = {}
_orig_parser3e = _search_args_from_text
def _capture_parser3e(text: str) -> dict:
    _captured3e["text"] = text
    return _orig_parser3e(text)
_search_args_from_text = _capture_parser3e
try:
    _s3e = ShopState(); _t3e = ToolTracer(); _h3e = []
    _ = run_memory_agent("I need running shoes", _s3e, TOOLS, _t3e, _h3e, _p3e)
finally:
    _search_args_from_text = _orig_parser3e
assert _captured3e["text"] == "I need running shoes"
print("OK 3.E: profile category check is case-insensitive")

# [3.F] Edge-case: explicit user brand overrides stored profile brand
_p3f = Path("_demo_profile_3f.json")
save_profile({"brand": "Sony", "max_price": "200", "category": "earbuds"}, _p3f)
_s3f = ShopState(); _t3f = ToolTracer(); _h3f = []
_r3f, _h3f = run_memory_agent("Find Apple earbuds under 300 dollars", _s3f, TOOLS, _t3f, _h3f, _p3f)
_search_call3f = _t3f.get_calls("search_products")[0]
assert _search_call3f.args["brand"] == "Apple"
print("OK 3.F: explicit brand intent is preserved")

# [3.G] Edge-case: profile update flow should add one HumanMessage
_p3g = Path("_demo_profile_3g.json")
if _p3g.exists(): _p3g.unlink()
_s3g = ShopState(); _t3g = ToolTracer(); _h3g = []
_r3g, _h3g = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s3g, TOOLS, _t3g, _h3g, _p3g
)
_human_count3g = sum(1 for m in _h3g if isinstance(m, HumanMessage))
assert _human_count3g == 1
print("OK 3.G: no duplicated HumanMessage during profile updates")

# [3.H] AIRankerAgent chooses model-selected candidate (mocked LLM)
def _mock_ranker_llm(_messages):
    return AIMessage(content='{"product_id": "p7", "reason": "better portable value"}')
_ctx3h = AgentContext(query="Need a compact mouse", candidates=[
    {"id": "p6", "name": "MX Master 3S", "price": 109, "rating": 4.8, "tags": ["wireless", "productivity"]},
    {"id": "p7", "name": "Pebble 2", "price": 34, "rating": 4.2, "tags": ["wireless", "portable"]},
])
_tr3h = ToolTracer()
_ctx3h = AIRankerAgent(chat_fn=_mock_ranker_llm).run(_ctx3h, _tr3h)
assert _ctx3h.best is not None and _ctx3h.best["id"] == "p7" and _tr3h.called("rank_candidates_ai")
print("OK 3.H: AIRankerAgent uses model decision")

# [3.I] AIRankerAgent fallback is deterministic on invalid LLM output
def _mock_ranker_llm_bad(_messages):
    return AIMessage(content="invalid output")
_ctx3i = AgentContext(query="Need a great mouse", candidates=[
    {"id": "p6", "name": "MX Master 3S", "price": 109, "rating": 4.8, "tags": ["wireless", "productivity"]},
    {"id": "p7", "name": "Pebble 2", "price": 34, "rating": 4.2, "tags": ["wireless", "portable"]},
])
_tr3i = ToolTracer()
_ctx3i = AIRankerAgent(chat_fn=_mock_ranker_llm_bad).run(_ctx3i, _tr3i)
assert _ctx3i.best is not None and _ctx3i.best["id"] == "p6" and _tr3i.called("rank_candidates_ai_fallback")
print("OK 3.I: AIRankerAgent fallback works")

# [3.J] CoordinatorAgent can use AIRankerAgent via flag
_s3j = ShopState()
_res3j = CoordinatorAgent(use_ai_ranker=True, ai_chat_fn=_mock_ranker_llm).run(
    "Find a wireless mouse under 120 dollars", _s3j, TOOLS
)
assert "delegate_ranker" in _res3j.trace
assert _res3j.context.best is not None and _res3j.context.best["id"] == "p7"
print("OK 3.J: CoordinatorAgent uses AIRankerAgent via flag")


OK 3.A
OK 3.B
OK 3.C
OK 3.D: context passed correctly, max_price is respected
OK 3.E: profile category check is case-insensitive
OK 3.F: explicit brand intent is preserved
OK 3.G: no duplicated HumanMessage during profile updates
OK 3.H: AIRankerAgent uses model decision
OK 3.I: AIRankerAgent fallback works
OK 3.J: CoordinatorAgent uses AIRankerAgent via flag
